# Silver Transformation
======================

Execute raw → silver transformations using the same logic as the pipeline.

In [ ]:
# Import from silver_transform.py
import sys
sys.path.insert(0, '..')

from scripts.silver_transform import (
    get_minio_client,
    get_all_objects,
    read_parquet_from_s3,
    transform_customers,
    transform_claims,
    run_silver_transform,
)

print("Imported silver transformation functions:")
print(f"  - get_minio_client")
print(f"  - get_all_objects")
print(f"  - read_parquet_from_s3")
print(f"  - transform_customers")
print(f"  - transform_claims")
print(f"  - run_silver_transform")

## Step 1: Inspect Raw Data

In [ ]:
# Inspect raw data before transformation
import os
import boto3
import pandas as pd

MINIO_CONFIG = {
    "endpoint": os.getenv("MINIO_ENDPOINT", "localhost:9900"),
    "access_key": os.getenv("MINIO_ROOT_USER", "minioadmin"),
    "secret_key": os.getenv("MINIO_ROOT_PASSWORD", "minioadmin"),
    "bucket": os.getenv("MINIO_BUCKET", "insurance-data"),
}

s3 = boto3.client(
    "s3",
    endpoint_url=f"http://{MINIO_CONFIG['endpoint']}",
    aws_access_key_id=MINIO_CONFIG["access_key"],
    aws_secret_access_key=MINIO_CONFIG["secret_key"],
)

def list_raw_objects(prefix: str) -> list:
    response = s3.list_objects_v2(Bucket=MINIO_CONFIG["bucket"], Prefix=prefix)
    return [obj["Key"] for obj in response.get("Contents", [])]

def read_parquet(key: str) -> pd.DataFrame:
    local_path = f"/tmp/{os.path.basename(key)}"
    s3.download_file(MINIO_CONFIG["bucket"], key, local_path)
    df = pd.read_parquet(local_path)
    os.unlink(local_path)
    return df

print("=== Raw PostgreSQL Customers ===")
pg_cust_keys = list_raw_objects("raw/customers/")
print(f"Files: {len(pg_cust_keys)}")
for k in pg_cust_keys[:3]:
    print(f"  {k}")

print("\n=== Raw PostgreSQL Claims ===")
pg_claim_keys = list_raw_objects("raw/claims/")
print(f"Files: {len(pg_claim_keys)}")
for k in pg_claim_keys[:3]:
    print(f"  {k}")

print("\n=== Raw Kaggle Customers ===")
kg_cust_keys = list_raw_objects("raw/kaggle_customers/")
print(f"Files: {len(kg_cust_keys)}")
for k in kg_cust_keys[:3]:
    print(f"  {k}")

print("\n=== Raw Kaggle Claims ===")
kg_claim_keys = list_raw_objects("raw/kaggle_claims/")
print(f"Files: {len(kg_claim_keys)}")
for k in kg_claim_keys[:3]:
    print(f"  {k}")

## Step 2: Load and Preview Raw Data

In [ ]:
# Load and preview raw PostgreSQL customers
if pg_cust_keys:
    df_pg_cust = read_parquet(pg_cust_keys[0])
    print(f"=== Raw PG Customers: {len(df_pg_cust)} rows ===")
    print(df_pg_cust.head(3))
    print("\nColumns:", list(df_pg_cust.columns))
else:
    print("No raw PG customer files found")

# Load and preview raw PostgreSQL claims
if pg_claim_keys:
    df_pg_claim = read_parquet(pg_claim_keys[0])
    print(f"\n=== Raw PG Claims: {len(df_pg_claim)} rows ===")
    print(df_pg_claim.head(3))
    print("\nColumns:", list(df_pg_claim.columns))
else:
    print("No raw PG claim files found")

In [ ]:
# Load and preview raw Kaggle data
if kg_cust_keys:
    df_kg_cust = read_parquet(kg_cust_keys[0])
    print(f"=== Raw Kaggle Customers: {len(df_kg_cust)} rows ===")
    print(df_kg_cust.head(3))
    print("\nColumns:", list(df_kg_cust.columns))
else:
    print("No raw Kaggle customer files found")

if kg_claim_keys:
    df_kg_claim = read_parquet(kg_claim_keys[0])
    print(f"\n=== Raw Kaggle Claims: {len(df_kg_claim)} rows ===")
    print(df_kg_claim.head(3))
    print("\nColumns:", list(df_kg_claim.columns))
else:
    print("No raw Kaggle claim files found")

## Step 3: Run Silver Transformation

In [ ]:
# Run the silver transformation
# This executes transform_customers() and transform_claims()
# to unify raw data and apply transformations

print("Running silver transformation...")
print("(This may take a moment)\n")

# Run transformation for customers
try:
    transform_customers()
    print("✓ Customers transformed")
except Exception as e:
    print(f"✗ Customer transformation failed: {e}")

# Run transformation for claims
try:
    transform_claims()
    print("✓ Claims transformed")
except Exception as e:
    print(f"✗ Claim transformation failed: {e}")

## Step 4: Verify Silver Output

In [ ]:
# Check silver output
print("=== Silver Customers ===")
silver_cust_keys = list_raw_objects("silver/silver_customers/")
print(f"Files: {len(silver_cust_keys)}")
for k in silver_cust_keys[-3:]:
    print(f"  {k}")

print("\n=== Silver Claims ===")
silver_claim_keys = list_raw_objects("silver/silver_claims/")
print(f"Files: {len(silver_claim_keys)}")
for k in silver_claim_keys[-3:]:
    print(f"  {k}")

In [ ]:
# Preview silver data
if silver_cust_keys:
    df_silver_cust = read_parquet(silver_cust_keys[-1])
    print(f"=== Silver Customers: {len(df_silver_cust)} rows ===")
    print(df_silver_cust.head(3))
    print("\nColumns:", list(df_silver_cust.columns))
    if "risk_bucket" in df_silver_cust.columns:
        print("\nRisk bucket distribution:")
        print(df_silver_cust["risk_bucket"].value_counts())
    if "source_system" in df_silver_cust.columns:
        print("\nSource system distribution:")
        print(df_silver_cust["source_system"].value_counts())

In [ ]:
# Preview silver claims
if silver_claim_keys:
    df_silver_claim = read_parquet(silver_claim_keys[-1])
    print(f"=== Silver Claims: {len(df_silver_claim)} rows ===")
    print(df_silver_claim.head(3))
    print("\nColumns:", list(df_silver_claim.columns))
    if "claim_status_category" in df_silver_claim.columns:
        print("\nClaim status category distribution:")
        print(df_silver_claim["claim_status_category"].value_counts())
    if "vehicle_category" in df_silver_claim.columns:
        print("\nVehicle category distribution:")
        print(df_silver_claim["vehicle_category"].value_counts())
    if "source_system" in df_silver_claim.columns:
        print("\nSource system distribution:")
        print(df_silver_claim["source_system"].value_counts())

## Step 5: Compare Raw vs Silver

In [ ]:
# Compare row counts
print("=== Row Counts ===\n")

# Raw counts
total_raw_cust = sum(len(read_parquet(k)) for k in pg_cust_keys) if pg_cust_keys else 0
total_raw_cust += sum(len(read_parquet(k)) for k in kg_cust_keys) if kg_cust_keys else 0
print(f"Raw Customers: {total_raw_cust}")

total_raw_claim = sum(len(read_parquet(k)) for k in pg_claim_keys) if pg_claim_keys else 0
total_raw_claim += sum(len(read_parquet(k)) for k in kg_claim_keys) if kg_claim_keys else 0
print(f"Raw Claims: {total_raw_claim}")

# Silver counts (latest files)
if silver_cust_keys:
    df_sc = read_parquet(silver_cust_keys[-1])
    print(f"\nSilver Customers: {len(df_sc)}")
if silver_claim_keys:
    df_scl = read_parquet(silver_claim_keys[-1])
    print(f"Silver Claims: {len(df_scl)}")